In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
# Notebook is in: <project_root>/02_Features/
# So project root is one level up:
ROOT = Path.cwd().parent

RAW_CSV = ROOT / "01_Data" / "cadli_btcusd_1m_2021-06-01_to_2025-12-01.csv"
OUT_CSV = ROOT / "02_Features" / "cadli_btcusd_1m_features_2021-06-01_to_2025-12-01.csv"

print("ROOT    :", ROOT)
print("RAW_CSV :", RAW_CSV)
print("OUT_CSV :", OUT_CSV)

# ---------------------------------------------------------
# 0) Load raw data
# ---------------------------------------------------------
# Adjust column names if your CSV is different
df = pd.read_csv(
    RAW_CSV,
    parse_dates=["datetime"],  # assumes there is a 'datetime' column
)

df = df.sort_values("datetime").set_index("datetime")

# Sanity check
print(df.head())
print(df.columns)

# We expect at least:
# ["OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"] in df.columns


# ---------------------------------------------------------
# 1) Log returns (1m, 5m, 15m)
# ---------------------------------------------------------
# Each log return here compares the CURRENT close with a PAST close.
# This is safe: we are not peeking into the future.
#
# log_ret_1m[t]  = log(CLOSE[t] / CLOSE[t-1])
# log_ret_5m[t]  = log(CLOSE[t] / CLOSE[t-5])
# log_ret_15m[t] = log(CLOSE[t] / CLOSE[t-15])
#
# NOTE: These describe the *past* 1, 5, 15 minutes of price movement.
#       They are NOT the same as "future 15-minute target" and do not leak labels.
df["log_ret_1m"] = np.log(df["CLOSE"] / df["CLOSE"].shift(1))
df["log_ret_5m"] = np.log(df["CLOSE"] / df["CLOSE"].shift(5))
df["log_ret_15m"] = np.log(df["CLOSE"] / df["CLOSE"].shift(15))

# ---------------------------------------------------------
# 2) Volatility features (5m, 15m)
# ---------------------------------------------------------
# vol_5m[t]  = std of 1-minute log returns over the LAST 5 minutes
# vol_15m[t] = std of 1-minute log returns over the LAST 15 minutes
#
# Again, these are purely based on PAST data up to time t.
df["vol_5m"] = df["log_ret_1m"].rolling(5).std()
df["vol_15m"] = df["log_ret_1m"].rolling(15).std()

# ---------------------------------------------------------
# 3) Range features (% of open)
# ---------------------------------------------------------
# 1-minute range as fraction of open:
#   (HIGH - LOW) / OPEN
df["range_pct_1m"] = (df["HIGH"] - df["LOW"]) / df["OPEN"]

# 5-minute range as fraction of *current* open:
#   (max(HIGH over last 5m) - min(LOW over last 5m)) / current OPEN
# This is still using only past candles (rolling(5)), so no future leakage.
df["range_pct_5m"] = (df["HIGH"].rolling(5).max() - df["LOW"].rolling(5).min()) / df[
    "OPEN"
]

# ---------------------------------------------------------
# 4) Candle shape features
# ---------------------------------------------------------
# These describe the micro-structure of the CURRENT 1-minute candle only.
# They do not peek into future candles.

# Body size (close - open) relative to open
df["body_pct"] = (df["CLOSE"] - df["OPEN"]) / df["OPEN"]

# Upper wick = distance from max(OPEN, CLOSE) to HIGH, relative to OPEN
df["upper_wick_pct"] = (df["HIGH"] - df[["OPEN", "CLOSE"]].max(axis=1)) / df["OPEN"]

# Lower wick = distance from LOW to min(OPEN, CLOSE), relative to OPEN
df["lower_wick_pct"] = (df[["OPEN", "CLOSE"]].min(axis=1) - df["LOW"]) / df["OPEN"]

# Normalized body relative to total candle range.
# (Protect with + 1e-9 to avoid division by zero when HIGH == LOW.)
df["body_norm"] = (df["CLOSE"] - df["OPEN"]) / (df["HIGH"] - df["LOW"] + 1e-9)

# ---------------------------------------------------------
# 5) Volume features (relative volume)
# ---------------------------------------------------------
# volume_ratio_5m[t]  = current VOLUME[t] / average VOLUME over last 5 minutes
# volume_ratio_15m[t] = current VOLUME[t] / average VOLUME over last 15 minutes
#
# These are backward-looking (rolling mean over past minutes).
df["volume_ratio_5m"] = df["VOLUME"] / (df["VOLUME"].rolling(5).mean() + 1e-9)
df["volume_ratio_15m"] = df["VOLUME"] / (df["VOLUME"].rolling(15).mean() + 1e-9)

# ---------------------------------------------------------
# 6) Current 15-min window boundaries
# ---------------------------------------------------------
# For each 1-minute bar, we want to know which 15-minute window it belongs to.
# We define:
#   - window_start = floor timestamp down to 15-minute boundary
#   - window_end   = window_start + 15 minutes
#
# Example:
#   t = 14:07  → window_start = 14:00, window_end = 14:15
#   t = 14:14  → window_start = 14:00, window_end = 14:15
#   t = 14:16  → window_start = 14:15, window_end = 14:30
df["window_start"] = df.index.floor("15min")
df["window_end"] = df["window_start"] + pd.Timedelta("15min")

# Time since start of the current 15-min window, in minutes.
# Example: window_start=14:00
#   t=14:00 → t_since_start_min = 0
#   t=14:01 → 1
#   ...
#   t=14:14 → 14
df["t_since_start_min"] = (
    (df.index - df["window_start"]).dt.total_seconds() / 60.0
)

# Time until end of the current 15-min window, in minutes.
# Example: window_end=14:15
#   t=14:00 → t_to_end_min = 15
#   t=14:01 → 14
#   ...
#   t=14:14 → 1

df["t_to_end_min"] = (
    (df["window_end"] - df.index).dt.total_seconds() / 60.0
)

# ---------------------------------------------------------
# 7) Drop NaNs from rolling features
# ---------------------------------------------------------
# All rolling and shifted features introduce NaNs at the beginning:
#   - log_ret_* need past bars
#   - vol_* and rolling ranges need several past points
# We drop rows with ANY NaN so that the feature matrix is clean.
df = df.dropna()

print("Final feature columns:", df.columns)
print("Final shape:", df.shape)

df.head()

ROOT    : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project
RAW_CSV : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/01_Data/cadli_btcusd_1m_2021-06-01_to_2025-12-01.csv
OUT_CSV : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/02_Features/cadli_btcusd_1m_features_2021-06-01_to_2025-12-01.csv
                                   OPEN          HIGH           LOW  \
datetime                                                              
2021-06-01 00:00:00+00:00  37352.647027  37352.647027  37240.049594   
2021-06-01 00:01:00+00:00  37240.049594  37250.756149  37240.049594   
2021-06-01 00:02:00+00:00  37250.756149  37393.393453  37250.756149   
2021-06-01 00:03:00+00:00  37393.393453  37587.297396  37393.393453   
2021-06-01 00:04:00+00:00  37587.297396  37626.837693  3

,OPEN,HIGH,LOW,CLOSE,VOLUME,QUOTE_VOLUME,VOLUME_TOP_TIER,QUOTE_VOLUME_TOP_TIER,VOLUME_DIRECT,QUOTE_VOLUME_DIRECT,...,body_pct,upper_wick_pct,lower_wick_pct,body_norm,volume_ratio_5m,volume_ratio_15m,window_start,window_end,t_since_start_min,t_to_end_min
datetime,,,,,,,,,,,,,,,,,,,,,
2021-06-01 00:15:00+00:00,37596.978146,37596.978146,37588.295217,37588.295217,386.274550,1.454586e+07,202.381039,7.610078e+06,101.682819,3.819137e+06,...,-0.000231,0.0,0.0,-1.0,0.984124,0.666775,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,0.0,15.0
2021-06-01 00:16:00+00:00,37588.295217,37602.223775,37588.295217,37602.223775,280.578555,1.055610e+07,125.918843,4.733019e+06,59.350673,2.229784e+06,...,0.000371,0.0,0.0,1.0,0.812895,0.495613,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,1.0,14.0
2021-06-01 00:17:00+00:00,37602.223775,37602.223775,37576.860725,37576.860725,280.033526,1.052722e+07,155.500319,5.839973e+06,55.016749,2.066074e+06,...,-0.000675,0.0,0.0,-1.0,0.822139,0.513584,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,2.0,13.0
2021-06-01 00:18:00+00:00,37576.860725,37576.860725,37549.917342,37549.917342,381.019556,1.430533e+07,216.922688,8.140797e+06,91.409644,3.429427e+06,...,-0.000717,0.0,0.0,-1.0,1.086210,0.834134,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,3.0,12.0
2021-06-01 00:19:00+00:00,37549.917342,37549.917342,37548.286482,37548.286482,276.513990,1.039699e+07,101.151345,3.803355e+06,35.871824,1.345747e+06,...,-0.000043,0.0,0.0,-1.0,0.861726,0.659693,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,4.0,11.0


In [24]:
# ---------------------------------------------------------
# Save to CSV
# ---------------------------------------------------------
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV)

# Print summary
print("Saved features to:", OUT_CSV)
print(f"Number of rows   : {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("\nColumns:")
print(list(df.columns))

Saved features to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/02_Features/cadli_btcusd_1m_features_2021-06-01_to_2025-12-01.csv
Number of rows   : 2367346
Number of columns: 29

Columns:
['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'QUOTE_VOLUME', 'VOLUME_TOP_TIER', 'QUOTE_VOLUME_TOP_TIER', 'VOLUME_DIRECT', 'QUOTE_VOLUME_DIRECT', 'VOLUME_TOP_TIER_DIRECT', 'QUOTE_VOLUME_TOP_TIER_DIRECT', 'log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm', 'volume_ratio_5m', 'volume_ratio_15m', 'window_start', 'window_end', 't_since_start_min', 't_to_end_min']


In [25]:
# ---------------------------------------------------------
# 8) Window open/close + label per 15-min window
# ---------------------------------------------------------
# Resample CLOSE into 15-minute windows
close_15 = df["CLOSE"].resample("15min", label="left", closed="left")

windows = pd.DataFrame({
    "open_15": close_15.first(),
    "close_15": close_15.last(),
})

windows = windows.dropna(subset=["open_15", "close_15"])
windows["y_up"] = (windows["close_15"] > windows["open_15"]).astype(int)

# Join back to each minute based on window_start
df = df.join(windows[["open_15", "close_15", "y_up"]], on="window_start") # Joins window-level labels back to minute-level data
df = df.dropna(subset=["open_15", "close_15", "y_up"])
df["y_up"] = df["y_up"].astype(int)

# ---------------------------------------------------------
# 9) Position of current price relative to window open
# ---------------------------------------------------------

# log distance to window open (this is what you asked about)
df["log_rel_to_open"] = np.log(df["CLOSE"] / df["open_15"])

# optional: plain percentage distance
df["rel_to_open_pct"] = (df["CLOSE"] / df["open_15"]) - 1

# print(df[["CLOSE", "open_15", "log_rel_to_open", "rel_to_open_pct", "y_up"]].head())
print(f"Number of rows   : {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("\nColumns:")
print(list(df.columns))

df.head()


Number of rows   : 2367346
Number of columns: 34

Columns:
['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'QUOTE_VOLUME', 'VOLUME_TOP_TIER', 'QUOTE_VOLUME_TOP_TIER', 'VOLUME_DIRECT', 'QUOTE_VOLUME_DIRECT', 'VOLUME_TOP_TIER_DIRECT', 'QUOTE_VOLUME_TOP_TIER_DIRECT', 'log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm', 'volume_ratio_5m', 'volume_ratio_15m', 'window_start', 'window_end', 't_since_start_min', 't_to_end_min', 'open_15', 'close_15', 'y_up', 'log_rel_to_open', 'rel_to_open_pct']


,OPEN,HIGH,LOW,CLOSE,VOLUME,QUOTE_VOLUME,VOLUME_TOP_TIER,QUOTE_VOLUME_TOP_TIER,VOLUME_DIRECT,QUOTE_VOLUME_DIRECT,...,volume_ratio_15m,window_start,window_end,t_since_start_min,t_to_end_min,open_15,close_15,y_up,log_rel_to_open,rel_to_open_pct
datetime,,,,,,,,,,,,,,,,,,,,,
2021-06-01 00:15:00+00:00,37596.978146,37596.978146,37588.295217,37588.295217,386.274550,1.454586e+07,202.381039,7.610078e+06,101.682819,3.819137e+06,...,0.666775,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,0.0,15.0,37588.295217,37735.589397,1,0.000000,0.000000
2021-06-01 00:16:00+00:00,37588.295217,37602.223775,37588.295217,37602.223775,280.578555,1.055610e+07,125.918843,4.733019e+06,59.350673,2.229784e+06,...,0.495613,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,1.0,14.0,37588.295217,37735.589397,1,0.000370,0.000371
2021-06-01 00:17:00+00:00,37602.223775,37602.223775,37576.860725,37576.860725,280.033526,1.052722e+07,155.500319,5.839973e+06,55.016749,2.066074e+06,...,0.513584,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,2.0,13.0,37588.295217,37735.589397,1,-0.000304,-0.000304
2021-06-01 00:18:00+00:00,37576.860725,37576.860725,37549.917342,37549.917342,381.019556,1.430533e+07,216.922688,8.140797e+06,91.409644,3.429427e+06,...,0.834134,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,3.0,12.0,37588.295217,37735.589397,1,-0.001022,-0.001021
2021-06-01 00:19:00+00:00,37549.917342,37549.917342,37548.286482,37548.286482,276.513990,1.039699e+07,101.151345,3.803355e+06,35.871824,1.345747e+06,...,0.659693,2021-06-01 00:15:00+00:00,2021-06-01 00:30:00+00:00,4.0,11.0,37588.295217,37735.589397,1,-0.001065,-0.001064


In [26]:

# ---------------------------------------------------------
# 10) Save labeled features to CSV
# ---------------------------------------------------------

OUT_LABELED = ROOT / "02_Features" / "cadli_btcusd_1m_features_labels_2021-06-01_to_2025-12-01.csv"

OUT_LABELED.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_LABELED)

print("Saved labeled features to:", OUT_LABELED)
print(f"Number of rows   : {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("Rows:", df.shape[0], "Columns:", df.shape[1])

df.head()

Saved labeled features to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/02_Features/cadli_btcusd_1m_features_labels_2021-06-01_to_2025-12-01.csv
Number of rows   : 2367346
Number of columns: 34
Rows: 2367346 Columns: 34
